# Energy Transition & Electric Mobility in Portugal — Data Extraction

**Phase 0-1: Exploration & Extraction**

This notebook explores the REN Data Hub and E-REDES Open Data APIs, confirms their real response structure, and builds the extraction pipeline for historical daily electricity production data.

Once the extraction logic is validated here, the reusable functions will move into `src/extract_ren.py` and `src/extract_eredes.py`.

## Setup

In [9]:
import requests
import pandas as pd
import json
import time
import os
from datetime import date, timedelta
from dateutil.relativedelta import relativedelta

pd.set_option("display.max_columns", None)

## Phase 0 - Exploring the REN Data Hub API

No token or registration required. Before writing the extraction loop, inspect the raw response for a single day to confirm the exact field names and structure.

In [10]:
REN_BASE_URL = "https://servicebus.ren.pt/datahubapi/electricity"

test_params = {"culture": "en-US", "date": "2025-01-25"}

response = requests.get(
    f"{REN_BASE_URL}/ElectricityProductionBreakdownDaily", params=test_params
)
response.raise_for_status()

data = response.json()
print(json.dumps(data, indent=2)[:2000])  # preview the structure

{
  "xAxis": {
    "title": {
      "text": "Hour"
    },
    "categories": [
      "00:00",
      "00:15",
      "00:30",
      "00:45",
      "01:00",
      "01:15",
      "01:30",
      "01:45",
      "02:00",
      "02:15",
      "02:30",
      "02:45",
      "03:00",
      "03:15",
      "03:30",
      "03:45",
      "04:00",
      "04:15",
      "04:30",
      "04:45",
      "05:00",
      "05:15",
      "05:30",
      "05:45",
      "06:00",
      "06:15",
      "06:30",
      "06:45",
      "07:00",
      "07:15",
      "07:30",
      "07:45",
      "08:00",
      "08:15",
      "08:30",
      "08:45",
      "09:00",
      "09:15",
      "09:30",
      "09:45",
      "10:00",
      "10:15",
      "10:30",
      "10:45",
      "11:00",
      "11:15",
      "11:30",
      "11:45",
      "12:00",
      "12:15",
      "12:30",
      "12:45",
      "13:00",
      "13:15",
      "13:30",
      "13:45",
      "14:00",
      "14:15",
      "14:30",
      "14:45",
      "15:00",
      "

## Phase 0 - Exploring the E-REDES Open Data API

E-REDES runs on the OpenDataSoft platform. The typical v2.1 records endpoint pattern is:

`https://{domain}/api/explore/v2.1/catalog/datasets/{dataset_id}/records`

Not yet verified with a live call — test below and adjust if the response shape differs.

Known fields for the mobility dataset (from E-REDES docs): `Data`, `Hora`, `Distrito`, `Concelho`, `Freguesia`, `Energia Consumida` (kWh).

In [11]:
EREDES_BASE_URL = "https://e-redes.opendatasoft.com/api/explore/v2.1/catalog/datasets"
DATASET_ID = "consumo_horario_mobilidade_eletrica_11_a_18"

response = requests.get(f"{EREDES_BASE_URL}/{DATASET_ID}/records", params={"limit": 20})
response.raise_for_status()

eredes_data = response.json()
print(json.dumps(eredes_data, indent=2)[:2000])

{
  "total_count": 4863215,
  "results": [
    {
      "datahora": "2025-12-05T13:00:00+00:00",
      "data": "2025-12-05",
      "hora": "13:00",
      "distrito": "Viana do Castelo",
      "concelho": "Ponte de Lima",
      "freguesia": "Fornelos e Queijada",
      "energia_consumida": 0.0,
      "codigo_distrito": "16",
      "codigo_concelho": "1607",
      "codigo_freguesia": "160758",
      "group_name": "11_a_18",
      "dia": "05",
      "mes": "12",
      "ano": "2025"
    },
    {
      "datahora": "2025-12-07T09:00:00+00:00",
      "data": "2025-12-07",
      "hora": "09:00",
      "distrito": "Porto",
      "concelho": "P\u00f3voa de Varzim",
      "freguesia": "Laundos",
      "energia_consumida": 0.0,
      "codigo_distrito": "13",
      "codigo_concelho": "1313",
      "codigo_freguesia": "131308",
      "group_name": "11_a_18",
      "dia": "07",
      "mes": "12",
      "ano": "2025"
    },
    {
      "datahora": "2025-12-26T21:00:00+00:00",
      "data": "2025-12-26"

## Phase 1 - Extraction: REN daily production breakdown

Loop over a date range, fetch each day, and accumulate results into a single structure.

In [12]:
def fetch_ren_daily_breakdown(target_date, delay=0.5):
    """Fetch REN electricity production breakdown for a single date."""
    params = {"culture": "en-US", "date": target_date.isoformat()}
    response = requests.get(
        f"{REN_BASE_URL}/ElectricityProductionBreakdownDaily", params=params
    )
    time.sleep(delay)  # polite delay, no documented rate limit

    if response.status_code != 200:
        print(f"Failed for {target_date}: {response.status_code}")
        return None

    return response.json()


def build_date_range(start_date, end_date):
    """Generate a list of dates between start_date and end_date (inclusive)."""
    days = (end_date - start_date).days
    return [start_date + timedelta(days=i) for i in range(days + 1)]


END_DATE = date.today() - timedelta(
    days=1
)  # yesterday — REN's most recent daily data may lag by a day
START_DATE = END_DATE - relativedelta(
    years=3
)  # rolling 3-year window, always recent regardless of when this runs

date_range = build_date_range(START_DATE, END_DATE)
print(f"Fetching {len(date_range)} days of data ({START_DATE} to {END_DATE})...")

raw_results = []
failed_dates = []

for d in date_range:
    result = fetch_ren_daily_breakdown(d)
    if result is not None:
        raw_results.append({"date": d, "data": result})
    else:
        failed_dates.append(d)

print(f"Success: {len(raw_results)} | Failed: {len(failed_dates)}")

Fetching 1097 days of data (2023-09-10 to 2026-09-10)...
Success: 1097 | Failed: 0


**Note:** pulling ~3 years of daily data at 0.5s/request takes roughly 12-15 minutes. Test with a much smaller range first (e.g. last 30 days) before running the full historical pull.

## Save raw data

In [13]:
os.makedirs("../data/raw", exist_ok=True)

with open("../data/raw/ren_production_breakdown_raw.json", "w") as f:
    json.dump(raw_results, f, indent=2, default=str)

print(f"Saved {len(raw_results)} records to data/raw/ren_production_breakdown_raw.json")

Saved 1097 records to data/raw/ren_production_breakdown_raw.json


## Next steps

- Confirm the real JSON structure and adjust `fetch_ren_daily_breakdown` if needed
- Repeat this extraction pattern for `ElectricityMarketPricesDaily`
- Build the equivalent extraction loop for E-REDES mobility data (check if district groups beyond `_11_a_18` exist)
- Move to **Phase 2 — Cleaning**: flatten the raw JSON into a tidy DataFrame